In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Hybrid-Recommender").getOrCreate()  
# create Spark session

26/03/17 20:54:59 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/17 20:54:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 20:55:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql import SparkSession   # create spark session
from pyspark.sql.functions import col  # column operations

# load ratings dataset
ratings = spark.read.csv(
    "/home/rajesh/CSL7100/Assignment3/ml-latest-small/ratings.csv",
    header=True,
    inferSchema=True
)

ratings.show(2)


+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
+------+-------+------+---------+
only showing top 2 rows



In [3]:
print((ratings.count(), len(ratings.columns)))

(100836, 4)


In [4]:
# Load movies dataset using Spark
movies = spark.read.csv("/home/rajesh/CSL7100/Assignment3/ml-latest-small/movies.csv", header=True, inferSchema=True)  
# header=True reads column names, inferSchema automatically detects column types

movies.show(2)  # preview dataset

+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
|      2|  Jumanji (1995)|Adventure|Childre...|
+-------+----------------+--------------------+
only showing top 2 rows



In [5]:
print((movies.count(), len(movies.columns)))

(9742, 3)


In [6]:

from pyspark.sql.functions import col

spark.conf.set("spark.sql.shuffle.partitions", "50")  
# reduce shuffle partitions → faster execution

ratings = ratings.cache()  
movies = movies.cache()    
# cache frequently used data

In [7]:
top_movies = ratings.groupBy("movieId").count() \
    .filter(col("count") > 20) \
    .select("movieId")
# keep only popular movies → reduces computation

ratings_filtered = ratings.join(top_movies, "movieId")

In [8]:
ratings_filtered.show(5)


+-------+------+------+---------+
|movieId|userId|rating|timestamp|
+-------+------+------+---------+
|      1|     1|   4.0|964982703|
|      3|     1|   4.0|964981247|
|      6|     1|   4.0|964982224|
|     47|     1|   5.0|964983815|
|     50|     1|   5.0|964982931|
+-------+------+------+---------+
only showing top 5 rows



In [9]:
print((ratings_filtered.count(), len(ratings_filtered.columns)))

(66658, 4)


In [10]:
top_movies.show(5)

+-------+
|movieId|
+-------+
|   1580|
|   2366|
|   2470|
|   3273|
|     32|
+-------+
only showing top 5 rows



In [11]:
print((top_movies.count(), len(top_movies.columns)))

(1235, 1)


In [12]:
from pyspark.sql.functions import col, split
from pyspark.ml.feature import HashingTF, IDF


movies_cbf = movies.join(top_movies, "movieId") \
    .withColumn("genre_words", split(col("genres"), "\\|"))
# split genres into words


hashingTF = HashingTF(
    inputCol="genre_words",
    outputCol="rawFeatures",
    numFeatures=20
)

featurized = hashingTF.transform(movies_cbf)

idf = IDF(inputCol="rawFeatures", outputCol="features")

idf_model = idf.fit(featurized)
# train IDF model

tfidf_data = idf_model.transform(featurized).cache()
# final TF-IDF features

In [13]:
tfidf_data.show(3)

+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|movieId|               title|              genres|         genre_words|         rawFeatures|            features|
+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|   1580|Men in Black (a.k...|Action|Comedy|Sci-Fi|[Action, Comedy, ...|(20,[0,16,19],[1....|(20,[0,16,19],[0....|
|   2366|    King Kong (1933)|Action|Adventure|...|[Action, Adventur...|(20,[0,10,12],[2....|(20,[0,10,12],[1....|
|   2470|Crocodile Dundee ...|    Adventure|Comedy| [Adventure, Comedy]|(20,[10,19],[1.0,...|(20,[10,19],[1.34...|
+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 3 rows



In [14]:
user_id = 1

user_movies = ratings_filtered.filter(col("userId")==user_id) \
    .select("movieId","rating")
# get movies rated by user

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
import numpy as np

def cosine_sim(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2))) if np.linalg.norm(v1) and np.linalg.norm(v2) else 0.0
# cosine similarity function

cosine_udf = udf(cosine_sim, DoubleType())

from pyspark.sql.functions import broadcast

cbf_scores = broadcast(user_movies).join(tfidf_data, "movieId") \
    .alias("u").join(tfidf_data.alias("m")) \
    .select(
        col("m.movieId").alias("movieId"),
        cosine_udf(col("u.features"), col("m.features")).alias("cbf_score")
    )

In [15]:
user_movies.show(2)

+-------+------+
|movieId|rating|
+-------+------+
|      1|   4.0|
|      3|   4.0|
+-------+------+
only showing top 2 rows



In [16]:
cbf_scores.show(2)

[Stage 49:>                                                         (0 + 1) / 1]

+-------+-------------------+
|movieId|          cbf_score|
+-------+-------------------+
|   1580|                1.0|
|   2366|0.23096921117539645|
+-------+-------------------+
only showing top 2 rows



In [17]:

from pyspark.sql.functions import sum, sqrt

movie_pairs = ratings_filtered.alias("r1").join(
    ratings_filtered.alias("r2"),
    (col("r1.userId")==col("r2.userId")) &
    (col("r1.movieId") < col("r2.movieId"))
).select(
    col("r1.movieId").alias("movie1"),
    col("r2.movieId").alias("movie2"),
    col("r1.rating").alias("rating1"),
    col("r2.rating").alias("rating2")
)
movie_pairs.show(3)

+------+------+-------+-------+
|movie1|movie2|rating1|rating2|
+------+------+-------+-------+
|     1|  5060|    4.0|    5.0|
|     1|  3809|    4.0|    4.0|
|     1|  3793|    4.0|    5.0|
+------+------+-------+-------+
only showing top 3 rows



In [18]:
print((movie_pairs.count(), len(movie_pairs.columns)))

(9123357, 4)


In [19]:
# create reduced movie pairs

similarity = movie_pairs.groupBy("movie1","movie2").agg(
    (sum(col("rating1")*col("rating2")) /
     (sqrt(sum(col("rating1")*col("rating1"))) *
      sqrt(sum(col("rating2")*col("rating2"))))
    ).alias("cf_score")
).cache()


In [20]:
similarity.show(3)

[Stage 73:>                                                         (0 + 1) / 1]

+------+------+------------------+
|movie1|movie2|          cf_score|
+------+------+------------------+
|     1|  2797|0.9780744414230232|
|     1|  1208|0.9641869773819718|
|     1|  1136|0.9620771596411308|
+------+------+------------------+
only showing top 3 rows



In [21]:

cf_pred = user_movies.join(
    similarity,
    user_movies.movieId == similarity.movie1
).select(
    col("movie2").alias("movieId"),
    (col("rating") * col("cf_score")).alias("weighted_score"),
    col("cf_score")
)

cf_scores = cf_pred.groupBy("movieId").agg(
    (sum("weighted_score") / sum("cf_score")).alias("cf_score")
).cache()
# compute CF score for user

In [22]:
cf_pred.show(3)

+-------+------------------+------------------+
|movieId|    weighted_score|          cf_score|
+-------+------------------+------------------+
|   2797| 3.912297765692093|0.9780744414230232|
|   1208| 3.856747909527887|0.9641869773819718|
|   1136|3.8483086385645233|0.9620771596411308|
+-------+------------------+------------------+
only showing top 3 rows



In [23]:
cf_scores.show(3)

+-------+-----------------+
|movieId|         cf_score|
+-------+-----------------+
|   1580|4.370192868018795|
|   3273|4.410649056780743|
|   2470|4.391929557347677|
+-------+-----------------+
only showing top 3 rows



In [38]:
from pyspark.sql.functions import min, max

# normalize CF
cf_min = cf_scores.agg({"cf_score":"min"}).collect()[0][0]
cf_max = cf_scores.agg({"cf_score":"max"}).collect()[0][0]

cf_scores = cf_scores.withColumn(
    "cf_score",
    (col("cf_score") - cf_min) / (cf_max - cf_min)
)
cf_scores.show(3)

+-------+------------------+
|movieId|          cf_score|
+-------+------------------+
|   1580| 0.820562076098197|
|   3273|0.9102364515641211|
|   2470|0.8687431850932605|
+-------+------------------+
only showing top 3 rows



In [39]:
# normalize CBF (optional but good)
cbf_min = cbf_scores.agg({"cbf_score":"min"}).collect()[0][0]
cbf_max = cbf_scores.agg({"cbf_score":"max"}).collect()[0][0]

cbf_scores = cbf_scores.withColumn(
    "cbf_score",
    (col("cbf_score") - cbf_min) / (cbf_max - cbf_min)
)
cbf_scores.show(3)

+-------+-------------------+
|movieId|          cbf_score|
+-------+-------------------+
|   1580| 0.9999999999999998|
|   2366| 0.2309692111753964|
|   2470|0.15103856443569064|
+-------+-------------------+
only showing top 3 rows



In [40]:
training_data = ratings_filtered.join(cf_scores, "movieId") \
                               .join(cbf_scores, "movieId") \
                               .select("rating","cf_score","cbf_score") \
                               .sample(0.3)  
# sample data → faster training

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler = VectorAssembler(
    inputCols=["cf_score","cbf_score"],
    outputCol="features"
)

ml_data = assembler.transform(training_data)
# create feature vector

lr = LinearRegression(
    featuresCol="features",
    labelCol="rating"
)

model = lr.fit(ml_data)

26/03/17 21:17:48 WARN Instrumentation: [d828c60c] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

In [41]:
final_data = cf_scores.join(cbf_scores, "movieId")

final_data = assembler.transform(final_data)

predictions = model.transform(final_data)
# predict final hybrid score

watched = user_movies.select("movieId")


In [42]:
predictions.show(3)

[Stage 387:>                                                        (0 + 1) / 1]

+-------+------------------+-------------------+--------------------+------------------+
|movieId|          cf_score|          cbf_score|            features|        prediction|
+-------+------------------+-------------------+--------------------+------------------+
|   1580| 0.820562076098197| 0.9999999999999998|[0.82056207609819...| 3.585107639097207|
|   2366|0.8540018535406556| 0.2309692111753964|[0.85400185354065...| 3.630855768922258|
|   2470|0.8687431850932605|0.15103856443569064|[0.86874318509326...|3.6361007207042197|
+-------+------------------+-------------------+--------------------+------------------+
only showing top 3 rows



In [43]:
watched.show(3)

+-------+
|movieId|
+-------+
|      1|
|      3|
|      6|
+-------+
only showing top 3 rows



In [45]:
from pyspark.sql.functions import max

final_unique = predictions.groupBy("movieId") \
    .agg(max("prediction").alias("prediction"))
# keep only one score per movie

recommendations = final_unique.join(
    movies, "movieId"
)
# join with movie metadata

top_n = recommendations.orderBy(col("prediction").desc()).limit(10)
# final top N

top_n.show(truncate=False)

[Stage 418:=====================================================> (49 + 1) / 50]

+-------+------------------+------------------------------------------------------------------+----------------------------+
|movieId|prediction        |title                                                             |genres                      |
+-------+------------------+------------------------------------------------------------------+----------------------------+
|223    |3.6505094285023842|Clerks (1994)                                                     |Comedy                      |
|79592  |3.6497056992687846|Other Guys, The (2010)                                            |Action|Comedy               |
|66097  |3.649205631698595 |Coraline (2009)                                                   |Animation|Fantasy|Thriller  |
|3160   |3.648933954016042 |Magnolia (1999)                                                   |Drama                       |
|3072   |3.648729177313539 |Moonstruck (1987)                                                 |Comedy|Romance              |


In [46]:
print("CF weight:", model.coefficients[0])
print("CBF weight:", model.coefficients[1])

CF weight: 0.04349791642958704
CBF weight: -0.05759661358699772


In [47]:
top_n.show(3)

[Stage 433:=====================================================> (49 + 1) / 50]

+-------+------------------+--------------------+--------------------+
|movieId|        prediction|               title|              genres|
+-------+------------------+--------------------+--------------------+
|    223|3.6505094285023842|       Clerks (1994)|              Comedy|
|  79592|3.6497056992687846|Other Guys, The (...|       Action|Comedy|
|  66097| 3.649205631698595|     Coraline (2009)|Animation|Fantasy...|
+-------+------------------+--------------------+--------------------+
only showing top 3 rows



In [48]:
spark.stop()